# Climate Pipeline — Step by Step

ERA5 (Zenodo municipality-daily + ARCO grid gaps) → UF × epidemiological-week
physical features, one step at a time.

Mirrors `src/pipeline/climate/` exactly — this notebook unrolls `build_zenodo_coverage()`
so each transformation is visible. Runs on the committed fixtures in
`tests/fixtures/climate/` (wide mun×day sample + two municipalities), so it
executes in seconds instead of ~1 min on the full Zenodo/ARCO corpus.

**Pipeline:** load mun reference → daily physical units → rainy-day flag → epiweek → mun×EW aggregate → population-weighted UF → attach `source` → order

**Literature alignment** (see `articles/README.md`):

| Choice | Grounding |
|---|---|
| ERA5 / ERA5-Land (not INMET) | Zhu et al. 2025; Chen & Moraga 2025; Sebastianelli et al. 2024 |
| Grain = **27 UF × epi-week** | Chen & Moraga (closest match to this repo) |
| Population-weighted means | Zhu; Chen & Moraga |
| Sunday-start epi-week; mean temp/RH/pressure; sum precip; rainy days | shared pattern in `articles/README.md` |
| RH from T + dewpoint | Sebastianelli et al. |
| Rainy-day threshold ≈ 0.03 mm | Sebastianelli / related ERA5 pipelines |
| No climate lags in the table | Chen & Moraga put lags **in the model** |


## Setup

In [1]:
import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pipeline.climate.spec import (
    DEFAULT_RAINY_DAY_MM,
    EW,
    FEATURE_COLUMNS,
    OUTPUT_COLUMNS,
    PRECIP_TOT,
    SOURCE_ZENODO,
    STATE_ABBREV,
)
from pipeline.climate.steps import (
    add_epiweek,
    aggregate_mun_week,
    convert_physical_units,
    load_municipalities,
    mark_rainy_days,
    order_columns,
    pop_weighted_to_uf,
    relative_humidity,
)
from pipeline.epiweek import floor_to_sunday

pl.Config.set_tbl_rows(8)

FIXTURES = PROJECT_ROOT / "tests" / "fixtures" / "climate"
FU_PATH = PROJECT_ROOT / "data" / "epidemiological" / "br_federative_units.csv"
MUNI_SAMPLE = FIXTURES / "br_municipalities_sample.csv"
ZENODO_SAMPLE = FIXTURES / "zenodo_daily_sample.parquet"

DEFAULT_RAINY_DAY_MM, MUNI_SAMPLE.exists(), ZENODO_SAMPLE.exists()


(0.03, True, True)

## Literature checklist (what this notebook must prove)

1. **Units** match published feature sets (°C, mm, %, hPa) — not raw ERA5 Kelvin / metres.
2. **RH** is derived from temperature + dewpoint (August–Roche–Magnus), as in Sebastianelli.
3. **Rainy days** use the ~0.03 mm daily threshold before weekly counting.
4. **Epi weeks** are Sunday-anchored — same helper as SINAN / Google Trends.
5. **UF reduction** is population-weighted, not a simple mean of municipalities.
6. Output keys are `(ew, state_abbrev, source)` ready to join the case series.


## Step 1 — Municipality reference (population weights)

Zhu / Chen & Moraga reduce grid or municipality climate to admin units with
**population-weighted** means. The committed reference
`data/climate/br_municipalities.csv` carries IBGE 2024 population estimates and
centroids; the fixture keeps two municipalities (SP capital + RJ capital) so the
weighting arithmetic is inspectable.


In [2]:
munis = load_municipalities(MUNI_SAMPLE).collect()
munis


code_muni,state_code,state_abbrev,population,lat,lon
i64,i16,str,f64,f64,f64
3550308,35,"""SP""",1.1e7,-23.55,-46.63
3304557,33,"""RJ""",6e6,-22.9,-43.17


## Step 2 — Scan daily municipality frame (Zenodo-shaped)

Production Zenodo files are *long* (`code_muni`, `date`, `name`, `value`) and
`join_zenodo_daily` pivots the six dengue-relevant variables. The fixture is
already *wide* in Kelvin / metres / Pascals — the same schema that enters
`convert_physical_units` after the join.


In [3]:
daily_raw = pl.scan_parquet(ZENODO_SAMPLE)
daily_raw.collect()


code_muni,date,temp_min_k,temp_mean_k,temp_max_k,dewpoint_k,precip_m,pressure_pa
i64,date,f64,f64,f64,f64,f64,f64
3550308,2025-01-05,293.15,298.15,303.15,293.15,0.0,101000.0
3304557,2025-01-05,294.15,299.15,304.15,294.15,0.001,100500.0
3550308,2025-01-06,292.15,297.15,302.15,292.15,0.002,101100.0
3304557,2025-01-06,293.15,298.15,303.15,293.15,0.0,100600.0


## Step 3 — Convert physical units + derive RH and thermal range

| Raw ERA5 | Output | Rule |
|---|---|---|
| Kelvin | `temp_*` °C | − 273.15 |
| metres | `precip_tot` mm | × 1000 |
| Pascals | `pressure` hPa | / 100 |
| T + dewpoint | `rel_humid` % | August–Roche–Magnus, clipped [0, 100] |
| — | `thermal_range` | `temp_max − temp_min` |

When air temperature equals dewpoint, RH must be ≈ 100% (sanity check below).


In [4]:
daily = convert_physical_units(daily_raw).collect()
daily.select(
    "code_muni", "date",
    "temp_min", "temp_mean", "temp_max",
    "precip_tot", "rel_humid", "pressure", "thermal_range",
)


code_muni,date,temp_min,temp_mean,temp_max,precip_tot,rel_humid,pressure,thermal_range
i64,date,f64,f64,f64,f64,f64,f64,f64
3550308,2025-01-05,20.0,25.0,30.0,0.0,73.802512,1010.0,10.0
3304557,2025-01-05,21.0,26.0,31.0,1.0,73.970629,1005.0,10.0
3550308,2025-01-06,19.0,24.0,29.0,2.0,73.632874,1011.0,10.0
3304557,2025-01-06,20.0,25.0,30.0,0.0,73.802512,1006.0,10.0


In [5]:
# Literature sanity: T == Td → RH ≈ 100%
pl.DataFrame({"t": [25.0], "td": [25.0]}).select(
    relative_humidity(pl.col("t"), pl.col("td")).alias("rh_pct")
)


rh_pct
f64
100.0


## Step 4 — Mark rainy days (threshold 0.03 mm)

Daily precip ≥ **0.03 mm** counts as a rainy day before the weekly sum — the
threshold used across the ERA5 dengue feature sets summarised in
`articles/README.md`. Null precip stays a null flag (excluded from the count).


In [6]:
daily_flagged = mark_rainy_days(daily.lazy(), DEFAULT_RAINY_DAY_MM).collect()
daily_flagged.select("code_muni", "date", PRECIP_TOT, "_is_rainy")


code_muni,date,precip_tot,_is_rainy
i64,date,f64,bool
3550308,2025-01-05,0.0,false
3304557,2025-01-05,1.0,true
3550308,2025-01-06,2.0,true
3304557,2025-01-06,0.0,false


## Step 5 — Floor dates to Sunday epi weeks

Same shared helper as SINAN (`src/pipeline/epiweek.py`). Chen & Moraga model at
UF × epi-week; Borges et al. (*Sci Data*) and this project's GT search already
anchor on Sundays — one calendar rule for every source.


In [7]:
with_ew = add_epiweek(daily_flagged.lazy()).collect()
with_ew.select("code_muni", EW, "temp_mean", PRECIP_TOT, "_is_rainy")


code_muni,ew,temp_mean,precip_tot,_is_rainy
i64,date,f64,f64,bool
3550308,2025-01-05,25.0,0.0,false
3304557,2025-01-05,26.0,1.0,true
3550308,2025-01-05,24.0,2.0,true
3304557,2025-01-05,25.0,0.0,false


In [8]:
# Every ew must be a Sunday (ISO weekday 7)
assert with_ew.filter(pl.col(EW).dt.weekday() != 7).is_empty()
with_ew.select(EW).unique()


ew
date
2025-01-05


## Step 6 — Aggregate municipality × epi-week

Within each week: **mean** for temp / RH / pressure; **sum** precip; **count**
rainy days. Thermal range is recomputed from weekly min/max means.


In [9]:
mun_week = aggregate_mun_week(daily_flagged.lazy().pipe(add_epiweek)).collect()
mun_week


code_muni,ew,temp_min,temp_mean,temp_max,precip_tot,rel_humid,pressure,rainy_days,thermal_range
i64,date,f64,f64,f64,f64,f64,f64,f64,f64
3304557,2025-01-05,20.5,25.5,30.5,1.0,73.886571,1005.5,1.0,10.0
3550308,2025-01-05,19.5,24.5,29.5,2.0,73.717693,1010.5,1.0,10.0


## Step 7 — Population-weighted reduction to UF

$$
x_{\mathrm{UF},w} = \frac{\sum_m p_m\, x_{m,w}}{\sum_m p_m}
$$

Nulls stay null (ARCO `temp_min`/`temp_max` must not become zeros after weighting).
With the fixture (one mun per UF) the UF value equals the municipality value —
still exercises the join + group-by path used in production.


In [10]:
uf = pop_weighted_to_uf(mun_week.lazy(), load_municipalities(MUNI_SAMPLE)).collect()
uf


ew,state_abbrev,temp_min,temp_mean,temp_max,precip_tot,rel_humid,pressure,rainy_days,thermal_range
date,str,f64,f64,f64,f64,f64,f64,f64,f64
2025-01-05,"""RJ""",20.5,25.5,30.5,1.0,73.886571,1005.5,1.0,10.0
2025-01-05,"""SP""",19.5,24.5,29.5,2.0,73.717693,1010.5,1.0,10.0


In [11]:
# Explicit weighted mean on a toy two-mun UF (literature formula)
from datetime import date

toy = pl.DataFrame({
    "code_muni": [1, 2],
    EW: [date(2024, 1, 7), date(2024, 1, 7)],
    "temp_mean": [20.0, 30.0],
    "temp_min": [15.0, 25.0],
    "temp_max": [25.0, 35.0],
    "precip_tot": [0.0, 0.0],
    "rel_humid": [50.0, 50.0],
    "pressure": [1000.0, 1000.0],
    "rainy_days": [0.0, 0.0],
    "thermal_range": [10.0, 10.0],
})
toy_munis = pl.DataFrame({
    "code_muni": [1, 2],
    "state_code": [35, 35],
    STATE_ABBREV: ["SP", "SP"],
    "population": [1.0, 3.0],
    "lat": [0.0, 0.0],
    "lon": [0.0, 0.0],
})
weighted = pop_weighted_to_uf(toy.lazy(), toy_munis.lazy()).collect()
assert weighted["temp_mean"][0] == 27.5  # (20*1 + 30*3) / 4
weighted.select(EW, STATE_ABBREV, "temp_mean")


ew,state_abbrev,temp_mean
date,str,f64
2024-01-07,"""SP""",27.5


## Step 8 — Attach `source`, order columns

Zenodo (ERA5-Land mun) and ARCO (ERA5 0.25° grid) share this schema; `source`
flags the product mix for 2023–2024 gaps. ARCO leaves `temp_min`/`temp_max` null
(collected NetCDFs are daily means only) — documented in
`src/pipeline/climate/README.md`.


In [12]:
series = order_columns(uf.lazy(), SOURCE_ZENODO).collect()
assert tuple(series.columns) == OUTPUT_COLUMNS
series


ew,state_abbrev,source,temp_min,temp_mean,temp_max,precip_tot,rel_humid,pressure,rainy_days,thermal_range
date,str,str,f64,f64,f64,f64,f64,f64,f64,f64
2025-01-05,"""RJ""","""zenodo""",20.5,25.5,30.5,1.0,73.886571,1005.5,1.0,10.0
2025-01-05,"""SP""","""zenodo""",19.5,24.5,29.5,2.0,73.717693,1010.5,1.0,10.0


## Quick look — fixture UF×EW features

In [13]:
series.select(EW, STATE_ABBREV, "source", "temp_mean", PRECIP_TOT, "rel_humid", "rainy_days")


ew,state_abbrev,source,temp_mean,precip_tot,rel_humid,rainy_days
date,str,str,f64,f64,f64,f64
2025-01-05,"""RJ""","""zenodo""",25.5,1.0,73.886571,1.0
2025-01-05,"""SP""","""zenodo""",24.5,2.0,73.717693,1.0


## Optional — peek at the full local series

If you already ran `uv run arboili transform climate`, load the published
artefact and summarise coverage by `source`.


In [14]:
full_path = PROJECT_ROOT / "data" / "climate" / "era5" / "ERA5_UF_EW.parquet"
if full_path.exists():
    full = pl.read_parquet(full_path)
    print(f"rows={full.height:,}  ew={full[EW].min()} → {full[EW].max()}")
    print(full.group_by("source").len().sort("source"))
    print(
        "temp_min null rate:",
        full["temp_min"].null_count() / full.height,
    )
else:
    print("Full series not built yet — run: uv run arboili transform climate")


rows=18,036  ew=2013-12-29 → 2026-09-27
shape: (2, 2)
┌────────┬───────┐
│ source ┆ len   │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ arco   ┆ 3915  │
│ zenodo ┆ 14121 │
└────────┴───────┘
temp_min null rate: 0.21706586826347304


## Integrity checks

Same checks the CLI and test suite run. Expect `states` to **fail** on the
fixture (only SP and RJ). On the real transform output, all 27 UFs pass.


In [15]:
from pipeline.climate.integrity import run_all

for check in run_all(series, FU_PATH):
    print(check)


[PASS] schema: 11 columns in published order
[PASS] unique_keys: 2 rows, all keys distinct
[PASS] weeks_start_on_sunday: all ew dates are Sundays
[FAIL] states: never appear in the series: ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'TO']
[PASS] source_domain: sources ['zenodo']
[PASS] rh_bounds: rel_humid in [0, 100] or null
[PASS] rainy_days_bounds: rainy_days in [0, 7] or null
[PASS] precip_non_negative: precip_tot >= 0 or null
[PASS] temp_order: temp_min ≤ temp_mean ≤ temp_max when all present
[PASS] feature_nulls_allowed: every row has ≥1 non-null feature


## Running it for real

This notebook mirrors the Zenodo half of `transform()` in
`src/pipeline/climate/transform.py`. The CLI also walks ARCO monthly NetCDFs
(nearest-centroid → mun → same pop-weighted UF path) and writes
`ERA5_UF_EW.parquet` + `.csv.gz`:

```bash
uv run arboili transform climate
```

Methods write-up for the data paper: `src/pipeline/climate/README.md`.
